In [1]:
import sys
sys.path.append("..")
sys.path.append("../scripts")

In [15]:
from ms2maccs import MS2Data, MS2MACCS, collate_fn
from utils import calc_tanimoto
from matchms.importing import load_from_mgf
from tqdm import tqdm

import rdkit.Chem as Chem
from rdkit.Chem import MACCSkeys
import numpy as np

import torch

In [3]:
m = MS2MACCS(
    "../models/standard_model.pt",
    "../fp_bit_maps/fp_bit_map_H_p_mode.pkl",
    "../fp_bit_maps/fp_bit_map_H_n_mode.pkl",
    "cuda",
)

In [28]:
test_preds_p = m.predict("../ms2_data/test_specs_H_p_mode.mgf").to("cpu")
test_preds_n = m.predict("../ms2_data/test_specs_H_n_mode.mgf").to("cpu")

Processing test_specs_H_p_mode.mgf: 1484it [00:02, 527.33it/s]
Prediction: 100%|█████████████████████████████████████████████████████████████████| 1484/1484 [00:04<00:00, 352.40it/s]
Processing test_specs_H_n_mode.mgf: 272it [00:00, 748.83it/s]
Prediction: 100%|███████████████████████████████████████████████████████████████████| 272/272 [00:00<00:00, 341.70it/s]


In [29]:
def specs2maccs(specs):
    true_maccs = []
    for spec in specs:
        smiles = spec.get("smiles")
        mol = Chem.MolFromSmiles(smiles)
        maccs = torch.tensor(np.array(MACCSkeys.GenMACCSKeys(mol)), dtype=torch.float32).to("cpu")
        true_maccs.append(maccs)

    return torch.stack([maccs for maccs in true_maccs])        

In [30]:
test_spec_p = list(tqdm(load_from_mgf("../ms2_data/test_specs_H_p_mode.mgf")))
test_spec_n = list(tqdm(load_from_mgf("../ms2_data/test_specs_H_n_mode.mgf")))

1484it [00:01, 1149.41it/s]
272it [00:00, 1678.51it/s]


In [31]:
true_maccs_p = specs2maccs(test_spec_p)
true_maccs_n = specs2maccs(test_spec_n)

In [32]:
help(calc_tanimoto)

Help on function calc_tanimoto in module utils:

calc_tanimoto(pred_maccs, true_maccs)
    calculates the tanimoto score for a batch of two tensors



In [36]:
"Positive mode test score:", round(calc_tanimoto(test_preds_p, true_maccs_p), 3)

('Positive mode test score:', 0.575)

In [37]:
"Positive mode test score:", round(calc_tanimoto(test_preds_n, true_maccs_n), 3)

('Positive mode test score:', 0.588)